In [10]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.animation import FuncAnimation


In [11]:
def forward_kinematics(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return x, y

In [12]:
def inverse_kinematics(x, y, L1, L2, elbow="up"):
    r2 = x**2 + y**2
    c2 = (r2 - L1**2 - L2**2) / (2 * L1 * L2)
    c2 = np.clip(c2, -1.0, 1.0)

    s2 = np.sqrt(1 - c2**2)

    if elbow == "down":
        s2 = -s2

    theta2 = np.arctan2(s2, c2)

    theta1 = np.arctan2(y, x) - np.arctan2(
        L2 * np.sin(theta2),
        L1 + L2 * np.cos(theta2)
    )

    return theta1, theta2


In [13]:
def error(x_true, y_true, L1=100, L2=100, elbow="up"):
    theta1, theta2 = inverse_kinematics(x_true, y_true, L1, L2, elbow=elbow)
    x_pred, y_pred = forward_kinematics(theta1, theta2, L1, L2)
    
    err = np.hypot(x_true - x_pred, y_true - y_pred)

    return err

In [14]:
print(error(120, 60)) # error should be 0

0.0


In [15]:
# constants
L1, L2 = 100, 100
N = 10_000
rng = np.random.default_rng(seed=42)

# joint angles
theta1 = rng.uniform(-np.pi, np.pi, N)
theta2 = rng.uniform(0, np.pi, N)

In [16]:
x, y = forward_kinematics(theta1, theta2, L1, L2)
X = np.column_stack([x, y])
Y = np.column_stack([theta1, theta2])

print(f"X shape: {X.shape}, Y Shape: {Y.shape}")
print(X[0]) # sanity check
print(Y[0]) # sanity check

X shape: (10000, 2), Y Shape: (10000, 2)
[-81.4462145   24.14134496]
[1.72131662 2.264235  ]


In [17]:
rng = np.random.default_rng(seed=42)
indices = rng.permutation(N) # shuffle the 10,000 indexes randomly

train_end = int(0.70 * N) # 7000
val_end = int(0.85 * N) # 8500

train_idx = indices[:train_end] # first 7000 -> train
val_idx = indices[train_end:val_end] # next 1500 -> val
test_idx = indices[val_end:] # -> next 1500 -> test

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]
X_test, Y_test = X[test_idx], Y[test_idx]

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)
print(X_test.shape, Y_test.shape)

(7000, 2) (7000, 2)
(1500, 2) (1500, 2)
(1500, 2) (1500, 2)


In [18]:
def mean_error_calc(X_true, Y_pred, L1, L2):

    # inputs to fk
    x_true = X_true[:,0]
    y_true = X_true[:,1]

    theta1_pred = Y_pred[:,0]
    theta2_pred = Y_pred[:,1]

    x_pred, y_pred = forward_kinematics(theta1_pred, theta2_pred, L1, L2)

    return np.hypot(x_true - x_pred, y_true - y_pred).mean()

In [19]:
def fit_linear(X, Y):
    # turn X -> Xb where Xb is the design matrix by prepending a col of 1s to X

    Xb = np.column_stack([np.ones(len(X)), X]) # [1, Xn, Yn]
    W, *_ = np.linalg.lstsq(Xb, Y, rcond=None) # least square problem

    return W

def predict_linear(W, X):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ W


In [22]:
W = fit_linear(X_train, Y_train)

Y_pred_train = predict_linear(W, X_train)
Y_pred_val = predict_linear(W, X_val)
Y_pred_test = predict_linear(W, X_test)

train_err = mean_error_calc(X_train, Y_pred_train, L1, L2)
val_err = mean_error_calc(X_val,   Y_pred_val,   L1, L2)
test_err = mean_error_calc(X_test,  Y_pred_test,  L1, L2)

print(f"Linear baseline mean EE error")
print(f"train: {train_err:.3f} mm")
print(f"val: {val_err:.3f} mm")
print(f"test: {test_err:.3f} mm")

Linear baseline mean EE error
train: 125.847 mm
val: 126.584 mm
test: 126.076 mm
